# 项目-航空AI助手

现在，我们将汇集我们所学到的知识，为航空公司打造人工智能客户支持助理

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
# 初始化

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

# 作为替代方案，如果您想使用 Ollama 而不是 OpenAI
# 检查 Ollama 是否在本地为您运行（请参阅 week1/day2 练习），然后取消注释接下来的 2 行
# 模型=“llama3.2”
# openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


In [ ]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

## 工具

工具是前沿法学硕士提供的一项极其强大的功能。

使用工具，您可以编写一个函数，并让 LLM 调用该函数作为其响应的一部分。

听起来很诡异……我们赋予它在我们的机器上运行代码的能力？

嗯，有点。

In [ ]:
# 让我们从创建一个有用的函数开始

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"The price of a ticket to {destination_city} is {price}"


In [ ]:
get_ticket_price("London")

In [ ]:
# There's a particular dictionary structure that's 必需 to describe our function:

get_price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [ ]:
# [Raga] 按照上面的示例添加了 set_ticket_price_function
# Used two properties destination city and price, made both of them 必需


set_price_function = {
    "name": "set_ticket_price",
    "description": "Set the price of a flight ticket to a given city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city for which the price will be set",
            },
            "price": {
                "type": "number",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city", "price"],
        "additionalProperties": False
    }
}

In [ ]:
# [Raga] 添加了 get_all_ticket_prices
# 克劳德的指示：
# 遵循与笔记本中的price_function相同的模式
# 为其提供适当的名称和描述（例如，“获取所有可用目的地的价格”）
# 重要提示：由于没有参数，参数仍应具有类型：“object”和属性：{}（空对象）
# Make 必需 an empty array []


get_all_ticket_prices_function = {
    "name": "get_all_ticket_prices",
    "description": "Get prices for all available destinationss.",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False
    }
}

In [ ]:
# 这包含在工具列表中：

tools = [{"type": "function", "function": get_price_function},{"type": "function", "function": set_price_function},{"type":"function","function":get_all_ticket_prices_function}]

In [ ]:
tools

## 让 OpenAI 使用我们的工具

有一些繁琐的东西可以让 OpenAI“调用我们的工具”

我们实际上做的是让法学硕士有机会通知我们它希望我们运行该工具。

新的聊天功能如下所示：

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)

      # 对于消息中的消息：
      # 打印（消息）
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [ ]:
# 我们必须编写该函数handle_tool_call：

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')
        price_details = get_ticket_price(city)
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    return response

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

## 让我们做一些改进

在 1 个响应中处理多个工具调用

处理多个工具调用一个接一个

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [ ]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [ ]:
import sqlite3


In [ ]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [ ]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [ ]:
get_ticket_price("London")

In [ ]:
def set_ticket_price(city, price):
    print(f"DATABASE TOOL CALLED: Setting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [ ]:
def get_all_ticket_prices():
    print ("DATABASE TOOL CALLED: Get all Ticket Prices", flush=True)
    with sqlite3.connect(DB) as conn: 
        cursor = conn.cursor()
        cursor.execute ('SELECT city, price FROM prices')
        result = cursor.fetchall()
        formatted = [f"{city.capitalize()}: ${price}" for city, price in result]
        return "Available ticket prices: " + ", ".join(formatted)

In [ ]:
print (get_all_ticket_prices())

In [ ]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [ ]:
get_ticket_price("Tokyo")

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

## 练习

添加一个工具来设置门票价格！

In [ ]:
# [Raga] 更新了此函数以包含 set_ticket_price

def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
        elif tool_call.function.name == "set_ticket_price": 
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price = arguments.get ('price')
            set_ticket_price(city,price)
            responses.append({
                "role": "tool",
                "content": f"Price set to {price} for city {city}",
                "tool_call_id": tool_call.id
            })
        elif tool_call.function.name == "get_all_ticket_prices":  #Added get all ticket prices
            all_prices = get_all_ticket_prices()
            responses.append({
                "role": "tool",
                "content": all_prices,
                "tool_call_id": tool_call.id
            })
         



    return responses

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="宽度：150px；高度：150px；垂直对齐：中间；">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">业务应用程序</h2>
            <span style="color:#181;">希望这几乎不需要说明！您现在可以向您的法学硕士采取行动。该航空公司助手现在不仅可以回答问题，还可以与预订 API 交互来进行预订！</span>
        </td>
    </tr>
</表>